# Crop AI — Benchmark v2: Deep Learning + Hybrid Ensemble + GradCAM++
**12-Method Synthesis Benchmark with 1D CNN, Deep Voting Ensemble, GradCAM++, LIME, SHAP**

**Author**: Adarsh, Bennett University

Sections:
1. Imports & Constants
2. Data Loading & Augmentation
3. EDA Figures (feature_distribution, feature_heatmap)
4. 10 Existing Models (RF, GB, Stacking, AdaBoost, SVM, MLP, XGB, TabNet, HybridDL+RF, Voting)
5. 1D CNN (New Model 11)
6. Deep Voting Ensemble (New Model 12)
7. GradCAM++ on 1D CNN
8. LIME + SHAP on RF500
9. Confusion Matrix
10. Save All Artifacts

## 0. Imports & Constants

In [ ]:
import os, sys, pickle, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# sklearn
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier, StackingClassifier,
                               AdaBoostClassifier, VotingClassifier)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

# XGBoost
from xgboost import XGBClassifier

# TabNet
from pytorch_tabnet.tab_model import TabNetClassifier

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# SHAP & LIME
import shap
import lime
import lime.lime_tabular

# Kaggle
import kagglehub
from kagglehub import KaggleDatasetAdapter

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

# ── Constants ──────────────────────────────────────────────────────────────────
SEED      = 42
FEATURES  = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
CROP_MAP  = {
    1:"rice",2:"maize",3:"jute",4:"cotton",5:"coconut",6:"papaya",7:"orange",
    8:"apple",9:"muskmelon",10:"watermelon",11:"grapes",12:"mango",13:"banana",
    14:"pomegranate",15:"lentil",16:"blackgram",17:"mungbean",18:"mothbeans",
    19:"pigeonpeas",20:"kidneybeans",21:"chickpea",22:"coffee"
}
NAME_TO_ID = {v: k for k, v in CROP_MAP.items()}
CROP_NAMES = [CROP_MAP[i+1] for i in range(22)]   # 0-indexed list

BASE = Path(__file__).parent.parent
ASSETS = BASE / "paper" / "assets"
MODEL_DIR = BASE / "model"
ASSETS.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Dark theme palette
P = {
    "bg":     "#0a0a0a",
    "panel":  "#111111",
    "border": "#1e1e1e",
    "text":   "#e0e0e0",
    "sub":    "#888888",
    "green":  "#22c55e",
    "blue":   "#3b82f6",
    "amber":  "#f59e0b",
    "red":    "#ef4444",
    "purple": "#a855f7",
    "teal":   "#14b8a6",
    "pink":   "#ec4899",
}

def dark_fig(w, h):
    fig, ax_or_axes = plt.subplots(figsize=(w, h), facecolor=P["bg"])
    return fig, ax_or_axes

def dark_axes(ax):
    ax.set_facecolor(P["panel"])
    for sp in ax.spines.values():
        sp.set_edgecolor(P["border"])
    ax.tick_params(colors=P["text"])
    ax.xaxis.label.set_color(P["text"])
    ax.yaxis.label.set_color(P["text"])
    ax.title.set_color(P["text"])

print("✅ Imports done.")

## 1. Data Loading & Augmentation

In [ ]:
print("Loading Kaggle dataset...")
raw_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "atharvaingle/crop-recommendation-dataset",
    "Crop_recommendation.csv"
)
print(f"  Loaded {len(raw_df)} samples, columns: {list(raw_df.columns)}")

# ── encode labels to 0-indexed integers ───────────────────────────────────────
raw_df["label_id"] = raw_df["label"].map(NAME_TO_ID) - 1   # 0..21
X_raw = raw_df[FEATURES].values.astype(np.float32)
y_raw = raw_df["label_id"].values.astype(np.int64)

# ── multivariate Gaussian augmentation (3×) ───────────────────────────────────
def augment_mvn(X, y, factor=3, seed=42):
    rng = np.random.default_rng(seed)
    X_parts, y_parts = [X], [y]
    for cls in np.unique(y):
        Xc = X[y == cls]
        mu  = Xc.mean(axis=0)
        cov = np.cov(Xc.T)
        n_new = len(Xc) * (factor - 1)
        Xn = rng.multivariate_normal(mu, cov, size=n_new).astype(np.float32)
        Xn = np.clip(Xn, X.min(axis=0), X.max(axis=0))
        X_parts.append(Xn)
        y_parts.append(np.full(n_new, cls, dtype=np.int64))
    return np.vstack(X_parts), np.concatenate(y_parts)

X_aug, y_aug = augment_mvn(X_raw, y_raw, factor=3)
print(f"  Augmented: {X_aug.shape[0]} samples (3×)")

# ── splits ────────────────────────────────────────────────────────────────────
X_tr,  X_te,  y_tr,  y_te  = train_test_split(X_aug, y_aug, test_size=0.2,
                                                random_state=SEED, stratify=y_aug)
X_tr0, X_te0, y_tr0, y_te0 = train_test_split(X_raw, y_raw, test_size=0.2,
                                                random_state=SEED, stratify=y_raw)

# ── scalers (fit on augmented train only) ─────────────────────────────────────
mm  = MinMaxScaler()
std = StandardScaler()
X_tr_s  = std.fit_transform(mm.fit_transform(X_tr))
X_te_s  = std.transform(mm.transform(X_te))
X_tr0_s = std.transform(mm.transform(X_tr0))
X_te0_s = std.transform(mm.transform(X_te0))

# save scalers for external scripts
with open(MODEL_DIR / "scalers.pkl", "wb") as f:
    pickle.dump({"mm": mm, "std": std}, f)
print("  Scalers saved to model/scalers.pkl")
print(f"  Train aug: {X_tr_s.shape} | Test aug: {X_te_s.shape}")
print(f"  Train raw: {X_tr0_s.shape} | Test raw: {X_te0_s.shape}")

## 2. EDA Figures

In [ ]:
print("\n=== EDA: Feature Distribution ===")

import seaborn as sns

# ── Figure 1: Feature Distribution (violin per feature, grouped by crop) ──────
# Use raw data (2200 samples, 22 crops) for clean distributions
df_raw = raw_df.copy()
df_raw["label_name"] = raw_df["label"]

crop_colors = plt.cm.tab20(np.linspace(0, 1, 22))

fig, axes = plt.subplots(2, 4, figsize=(22, 10), facecolor=P["bg"])
fig.suptitle("Feature Distribution by Crop (Kaggle Dataset, n=2200)",
             color=P["text"], fontsize=14, fontweight="bold", y=1.01)

for idx, feat in enumerate(FEATURES):
    ax = axes[idx // 4][idx % 4]
    ax.set_facecolor(P["panel"])
    for sp in ax.spines.values():
        sp.set_edgecolor(P["border"])

    data_by_crop = [df_raw[df_raw["label"] == CROP_MAP[i+1]][feat].values
                    for i in range(22)]
    bp = ax.boxplot(data_by_crop, patch_artist=True, showfliers=False,
                    medianprops=dict(color=P["amber"], linewidth=1.5),
                    whiskerprops=dict(color=P["sub"]),
                    capprops=dict(color=P["sub"]),
                    boxprops=dict(linewidth=0.8))
    for patch, col in zip(bp["boxes"], crop_colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.75)

    ax.set_xticks(range(1, 23))
    ax.set_xticklabels([CROP_MAP[i+1][:4] for i in range(22)],
                       rotation=90, fontsize=6, color=P["text"])
    ax.tick_params(colors=P["text"])
    units = ["mg/kg","mg/kg","mg/kg","°C","%","","mm"]
    ax.set_title(f"{feat} [{units[idx]}]", color=P["text"], fontsize=10, fontweight="bold")
    ax.set_facecolor(P["panel"])
    ax.grid(axis="y", color=P["border"], linewidth=0.4)

# hide last (8th) subplot
axes[1][3].set_visible(False)

plt.tight_layout()
out = ASSETS / "feature_distribution.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print(f"  Saved {out}")

# ── Figure 2: Feature Heatmap (Pearson correlation) ───────────────────────────
print("=== EDA: Feature Heatmap ===")

df_aug = pd.DataFrame(X_aug, columns=FEATURES)
corr = df_aug.corr()

fig, ax = plt.subplots(figsize=(9, 8), facecolor=P["bg"])
ax.set_facecolor(P["panel"])
for sp in ax.spines.values():
    sp.set_edgecolor(P["border"])

im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, label="Pearson r").ax.tick_params(colors=P["text"])

ax.set_xticks(range(7))
ax.set_yticks(range(7))
ax.set_xticklabels(FEATURES, rotation=45, ha="right", fontsize=10, color=P["text"])
ax.set_yticklabels(FEATURES, fontsize=10, color=P["text"])
ax.tick_params(colors=P["text"])

for i in range(7):
    for j in range(7):
        val = corr.values[i, j]
        tc = "white" if abs(val) > 0.6 else P["text"]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=9, color=tc, fontweight="bold")

ax.set_title("Feature Correlation Heatmap (Augmented Dataset, n=6600)",
             color=P["text"], fontsize=12, fontweight="bold", pad=12)
plt.tight_layout()
out = ASSETS / "feature_heatmap.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print(f"  Saved {out}")

## 3. Train 10 Existing Models

In [ ]:
def metrics(y_true, y_pred):
    return (accuracy_score(y_true, y_pred),
            precision_score(y_true, y_pred, average="macro", zero_division=0),
            recall_score(y_true, y_pred, average="macro", zero_division=0),
            f1_score(y_true, y_pred, average="macro", zero_division=0))

RESULTS_12 = {}

# helper: train/eval on both raw and augmented
def bench(name, clf_fn):
    print(f"  [{name}] training on raw...", end=" ", flush=True)
    c0 = clf_fn(); c0.fit(X_tr0_s, y_tr0)
    r0 = metrics(y_te0, c0.predict(X_te0_s))
    print(f"raw={r0[0]:.4f}  aug...", end=" ", flush=True)
    c1 = clf_fn(); c1.fit(X_tr_s, y_tr)
    r1 = metrics(y_te, c1.predict(X_te_s))
    print(f"aug={r1[0]:.4f}")
    RESULTS_12[name] = list(r0) + list(r1)
    return c1    # return augmented-trained model

print("\n=== Training 10 Existing Models ===")

# 1. Random Forest (100)
rf100 = bench("Random Forest", lambda: RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))

# 2. Gradient Boosting
bench("Gradient Boosting", lambda: GradientBoostingClassifier(n_estimators=100, random_state=SEED))

# 3. Stacking
def make_stacking():
    return StackingClassifier(
        estimators=[
            ("rf",  RandomForestClassifier(n_estimators=50, random_state=SEED, n_jobs=-1)),
            ("gb",  GradientBoostingClassifier(n_estimators=50, random_state=SEED)),
            ("svm", SVC(kernel="rbf", probability=True, random_state=SEED)),
        ],
        final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
        cv=3, n_jobs=-1
    )
bench("Stacking", make_stacking)

# 4. AdaBoost
bench("AdaBoost (SAMME)", lambda: AdaBoostClassifier(n_estimators=100, random_state=SEED))

# 5. SVM
bench("SVM (RBF)", lambda: SVC(kernel="rbf", probability=True, random_state=SEED))

# 6. MLP
bench("MLP (128,64)", lambda: MLPClassifier(hidden_layer_sizes=(128, 64),
      max_iter=300, random_state=SEED))

# 7. XGBoost
bench("XGBoost", lambda: XGBClassifier(n_estimators=100, random_state=SEED,
      eval_metric="mlogloss", verbosity=0))

# 8. TabNet
def bench_tabnet():
    print(f"  [TabNet] training on raw...", end=" ", flush=True)
    t0 = TabNetClassifier(verbose=0, seed=SEED)
    t0.fit(X_tr0_s, y_tr0, eval_set=[(X_te0_s, y_te0)], patience=15, max_epochs=100)
    r0 = metrics(y_te0, t0.predict(X_te0_s))
    print(f"raw={r0[0]:.4f}  aug...", end=" ", flush=True)
    t1 = TabNetClassifier(verbose=0, seed=SEED)
    t1.fit(X_tr_s, y_tr, eval_set=[(X_te_s, y_te)], patience=15, max_epochs=100)
    r1 = metrics(y_te, t1.predict(X_te_s))
    print(f"aug={r1[0]:.4f}")
    RESULTS_12["TabNet"] = list(r0) + list(r1)
    return t1

tabnet_model = bench_tabnet()

# 9. Hybrid DL+RF: MLP penultimate → RF
def make_hybrid_dl_rf(X_tr_in, y_tr_in, X_te_in, y_te_in, tag):
    mlp = MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=500,
                        random_state=SEED, early_stopping=True, validation_fraction=0.1)
    mlp.fit(X_tr_in, y_tr_in)
    # extract penultimate activations
    acts_tr = X_tr_in
    for coef, bias in zip(mlp.coefs_[:-1], mlp.intercepts_[:-1]):
        acts_tr = np.maximum(0, acts_tr @ coef + bias)
    acts_te = X_te_in
    for coef, bias in zip(mlp.coefs_[:-1], mlp.intercepts_[:-1]):
        acts_te = np.maximum(0, acts_te @ coef + bias)
    rf_h = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
    rf_h.fit(acts_tr, y_tr_in)
    return metrics(y_te_in, rf_h.predict(acts_te)), (mlp, rf_h)

print(f"  [Hybrid DL+RF] training on raw...", end=" ", flush=True)
r0, _ = make_hybrid_dl_rf(X_tr0_s, y_tr0, X_te0_s, y_te0, "raw")
print(f"raw={r0[0]:.4f}  aug...", end=" ", flush=True)
r1, hybrid_models = make_hybrid_dl_rf(X_tr_s, y_tr, X_te_s, y_te, "aug")
print(f"aug={r1[0]:.4f}")
RESULTS_12["Hybrid DL+RF"] = list(r0) + list(r1)

# 10. VotingClassifier (RF500 + ET500)
print(f"  [VotingClassifier(RF+ET)] training on raw...", end=" ", flush=True)
vc0 = VotingClassifier(
    estimators=[("rf", RandomForestClassifier(500, random_state=SEED, n_jobs=-1)),
                ("et", ExtraTreesClassifier(500, random_state=SEED, n_jobs=-1))],
    voting="soft", n_jobs=-1)
vc0.fit(X_tr0_s, y_tr0)
r0 = metrics(y_te0, vc0.predict(X_te0_s))
print(f"raw={r0[0]:.4f}  aug...", end=" ", flush=True)
vc1 = VotingClassifier(
    estimators=[("rf", RandomForestClassifier(500, random_state=SEED, n_jobs=-1)),
                ("et", ExtraTreesClassifier(500, random_state=SEED, n_jobs=-1))],
    voting="soft", n_jobs=-1)
vc1.fit(X_tr_s, y_tr)
r1 = metrics(y_te, vc1.predict(X_te_s))
print(f"aug={r1[0]:.4f}")
RESULTS_12["VotingClassifier\n(RF+ET)"] = list(r0) + list(r1)

print("\nExisting 10 models done.")
for k, v in RESULTS_12.items():
    print(f"  {k.replace(chr(10),' '):<30} before={v[0]:.4f}  after={v[4]:.4f}")

## 4. 1D CNN

In [ ]:
print("\n=== 1D CNN ===")

class CropCNN(nn.Module):
    def __init__(self, n_classes=22):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.1))
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.1))
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU())          # GradCAM++ hook here
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes))

    def forward(self, x):        # x: (B, 7)
        x = x.unsqueeze(1)       # → (B, 1, 7)
        x = self.conv1(x)        # → (B, 64, 7)
        x = self.conv2(x)        # → (B, 128, 7)
        x = self.conv3(x)        # → (B, 64, 7)
        x = self.pool(x).squeeze(-1)  # → (B, 64)
        return self.fc(x)             # → (B, 22)


def train_cnn(X_tr_in, y_tr_in, X_te_in, y_te_in, tag):
    device = "cpu"
    model  = CropCNN(22).to(device)
    opt    = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    crit   = nn.CrossEntropyLoss()

    Xtr_t = torch.tensor(X_tr_in, dtype=torch.float32)
    ytr_t = torch.tensor(y_tr_in, dtype=torch.long)
    Xte_t = torch.tensor(X_te_in, dtype=torch.float32)

    ds  = TensorDataset(Xtr_t, ytr_t)
    dl  = DataLoader(ds, batch_size=128, shuffle=True)

    best_acc, patience_cnt, best_state = 0.0, 0, None
    for epoch in range(150):
        model.train()
        for xb, yb in dl:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        sched.step()
        # validation
        model.eval()
        with torch.no_grad():
            logits = model(Xte_t)
            preds  = logits.argmax(1).numpy()
        acc = accuracy_score(y_te_in, preds)
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= 20:
            print(f"    [{tag}] early stop epoch {epoch}, best={best_acc:.4f}")
            break
    model.load_state_dict(best_state)
    return model, best_acc

print("  Training CNN on raw split...")
cnn_raw, cnn_raw_acc = train_cnn(X_tr0_s, y_tr0, X_te0_s, y_te0, "raw")
print(f"  CNN raw acc = {cnn_raw_acc:.4f}")

print("  Training CNN on augmented split...")
cnn_model, cnn_aug_acc = train_cnn(X_tr_s, y_tr, X_te_s, y_te, "aug")
print(f"  CNN aug acc = {cnn_aug_acc:.4f}")

# full metrics
cnn_model.eval()
with torch.no_grad():
    preds_cnn_aug = cnn_model(torch.tensor(X_te_s, dtype=torch.float32)).argmax(1).numpy()
preds_cnn_raw = cnn_raw.eval() or None
cnn_raw.eval()
with torch.no_grad():
    preds_cnn_raw = cnn_raw(torch.tensor(X_te0_s, dtype=torch.float32)).argmax(1).numpy()

r_cnn_raw = metrics(y_te0, preds_cnn_raw)
r_cnn_aug = metrics(y_te,  preds_cnn_aug)
RESULTS_12["1D CNN"] = list(r_cnn_raw) + list(r_cnn_aug)
print(f"  1D CNN: before={r_cnn_raw[0]:.4f}  after={r_cnn_aug[0]:.4f}")

# save CNN
torch.save(cnn_model.state_dict(), MODEL_DIR / "cnn_model.pt")
print(f"  CNN saved → model/cnn_model.pt")

## 5. Deep Voting Ensemble

In [ ]:
print("\n=== Deep Voting Ensemble (RF500 + ET500 + XGB + CNN) ===")

# Train components on augmented data (0-indexed labels)
print("  Training RF500...", end=" ", flush=True)
rf500 = RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1)
rf500.fit(X_tr_s, y_tr)
print(f"acc={accuracy_score(y_te, rf500.predict(X_te_s)):.4f}")

print("  Training ET500...", end=" ", flush=True)
et500 = ExtraTreesClassifier(n_estimators=500, random_state=SEED, n_jobs=-1)
et500.fit(X_tr_s, y_tr)
print(f"acc={accuracy_score(y_te, et500.predict(X_te_s)):.4f}")

print("  Training XGB300...", end=" ", flush=True)
xgb300 = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                        random_state=SEED, eval_metric="mlogloss", verbosity=0,
                        n_jobs=-1)
xgb300.fit(X_tr_s, y_tr)
print(f"acc={accuracy_score(y_te, xgb300.predict(X_te_s)):.4f}")

# save components
with open(MODEL_DIR / "rf500.pkl",  "wb") as f: pickle.dump(rf500,  f)
with open(MODEL_DIR / "et500.pkl",  "wb") as f: pickle.dump(et500,  f)
with open(MODEL_DIR / "xgb300.pkl", "wb") as f: pickle.dump(xgb300, f)

def deep_voting_proba(X_scaled):
    """Soft average of RF500 + ET500 + XGB300 + CNN probabilities."""
    p_rf  = rf500.predict_proba(X_scaled)           # (N,22)
    p_et  = et500.predict_proba(X_scaled)           # (N,22)
    p_xgb = xgb300.predict_proba(X_scaled)          # (N,22)
    cnn_model.eval()
    with torch.no_grad():
        logits = cnn_model(torch.tensor(X_scaled.astype(np.float32)))
        p_cnn  = torch.softmax(logits, dim=1).numpy()   # (N,22)
    return (p_rf + p_et + p_xgb + p_cnn) / 4.0

def deep_voting_predict(X_scaled):
    return deep_voting_proba(X_scaled).argmax(axis=1)

# eval on raw split
print("  Evaluating on raw split...", end=" ", flush=True)
# for "before" augmentation, all components re-trained on raw:
rf500_r = RandomForestClassifier(500, random_state=SEED, n_jobs=-1); rf500_r.fit(X_tr0_s, y_tr0)
et500_r = ExtraTreesClassifier(500, random_state=SEED, n_jobs=-1); et500_r.fit(X_tr0_s, y_tr0)
xgb_r   = XGBClassifier(300, max_depth=6, learning_rate=0.05, random_state=SEED,
                         eval_metric="mlogloss", verbosity=0, n_jobs=-1)
xgb_r.fit(X_tr0_s, y_tr0)

def dv_raw_predict(X_scaled):
    cnn_raw.eval()
    with torch.no_grad():
        logits = cnn_raw(torch.tensor(X_scaled.astype(np.float32)))
        p_cnn  = torch.softmax(logits, dim=1).numpy()
    p_avg = (rf500_r.predict_proba(X_scaled) + et500_r.predict_proba(X_scaled) +
             xgb_r.predict_proba(X_scaled) + p_cnn) / 4.0
    return p_avg.argmax(axis=1)

r_dv_raw = metrics(y_te0, dv_raw_predict(X_te0_s))
print(f"before={r_dv_raw[0]:.4f}")

print("  Evaluating on augmented split...", end=" ", flush=True)
r_dv_aug = metrics(y_te, deep_voting_predict(X_te_s))
print(f"after={r_dv_aug[0]:.4f}")

RESULTS_12["Deep Voting\nEnsemble"] = list(r_dv_raw) + list(r_dv_aug)
print(f"\n  Deep Voting Ensemble: before={r_dv_raw[0]:.4f}  after={r_dv_aug[0]:.4f}")
print(f"  (Baseline VotingClassifier(RF+ET): {RESULTS_12['VotingClassifier\\n(RF+ET)'][4]:.4f})")

if r_dv_aug[0] > RESULTS_12["VotingClassifier\n(RF+ET)"][4]:
    print("  ✅ NEW SOTA — Deep Voting Ensemble beats VotingClassifier(RF+ET)!")
else:
    print("  ⚠️  Did not beat VotingClassifier(RF+ET) — check hyperparameters")

## 6. GradCAM++ on 1D CNN

In [ ]:
print("\n=== GradCAM++ ===")

class GradCAMPlusPlus:
    """GradCAM++ for 1D CNN with GlobalAvgPool.

    Registers hooks on the last Conv1d layer (conv3[0]).
    Returns per-input-feature attribution scores in [0,1].
    """
    def __init__(self, model):
        self.model       = model
        self.activations = None
        self.gradients   = None
        # hook on the inner Conv1d of conv3 Sequential
        target = model.conv3[0]   # nn.Conv1d(128→64)
        self._fwd = target.register_forward_hook(self._save_activations)
        self._bwd = target.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach()          # (1, 64, 7)

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()    # (1, 64, 7)

    def remove(self):
        self._fwd.remove(); self._bwd.remove()

    def compute(self, x_scaled_1d, target_class):
        """x_scaled_1d: np.array (7,) | target_class: int 0..21
        Returns: attribution np.array (7,) in [0,1]
        """
        self.model.eval()
        x = torch.tensor(x_scaled_1d, dtype=torch.float32).unsqueeze(0)  # (1,7)
        logits = self.model(x)                                             # (1,22)
        self.model.zero_grad()
        oh = torch.zeros_like(logits); oh[0, target_class] = 1.0
        logits.backward(gradient=oh)

        A = self.activations.squeeze(0)    # (64, 7)
        G = self.gradients.squeeze(0)      # (64, 7)
        alpha = torch.relu(G.mean(dim=1))  # (64,)
        cam = (alpha.unsqueeze(1) * A).sum(dim=0)  # (7,)
        cam = torch.relu(cam)
        if cam.max() - cam.min() > 1e-8:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam.numpy()

gradcam = GradCAMPlusPlus(cnn_model)

# Compute attributions for 4 representative crops
TARGET_CROPS = {"rice": 0, "cotton": 3, "mango": 11, "coffee": 21}
gradcam_attrs = {}
for crop_name, crop_idx in TARGET_CROPS.items():
    mask = (y_te == crop_idx)
    # find first correctly-classified test sample
    cnn_model.eval()
    preds_all = preds_cnn_aug   # already computed
    correct_mask = mask & (preds_all == crop_idx)
    if correct_mask.sum() == 0:
        correct_mask = mask          # fallback: any sample of this crop
    sample = X_te_s[correct_mask][0]
    attr = gradcam.compute(sample, crop_idx)
    gradcam_attrs[crop_name] = attr
    top = FEATURES[attr.argmax()]
    print(f"  {crop_name:<12} top feature: {top:<15} "
          f"attrs: {dict(zip(FEATURES, attr.round(3)))}")

gradcam.remove()

# save
with open(ASSETS / "gradcam_attrs.pkl", "wb") as f:
    pickle.dump(gradcam_attrs, f)

# ── Figure: gradcam_crops.png ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=P["bg"])
fig.suptitle("GradCAM++ Feature Attribution — 1D CNN",
             color=P["text"], fontsize=14, fontweight="bold")

crop_titles = {"rice": "Rice", "cotton": "Cotton", "mango": "Mango", "coffee": "Coffee"}
for ax, (cn, attr) in zip(axes.flat, gradcam_attrs.items()):
    ax.set_facecolor(P["panel"])
    for sp in ax.spines.values(): sp.set_edgecolor(P["border"])
    colors = [P["green"] if v > 0.5 else P["blue"] for v in attr]
    bars = ax.bar(FEATURES, attr, color=colors, alpha=0.85, zorder=3)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Attribution Score [0–1]", color=P["text"], fontsize=9)
    ax.set_title(f"GradCAM++: {crop_titles[cn]}", color=P["text"],
                 fontsize=11, fontweight="bold")
    ax.tick_params(colors=P["text"], axis="both")
    ax.set_xticklabels(FEATURES, rotation=30, ha="right", fontsize=9, color=P["text"])
    ax.grid(axis="y", color=P["border"], linewidth=0.4, zorder=0)
    for bar, val in zip(bars, attr):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                f"{val:.2f}", ha="center", va="bottom",
                fontsize=7.5, color=P["text"])

plt.tight_layout()
out = ASSETS / "gradcam_crops.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print(f"  Saved {out}")

## 7. LIME + SHAP (on RF500)

In [ ]:
print("\n=== LIME ===")

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_tr_s,
    feature_names=FEATURES,
    class_names=CROP_NAMES,
    discretize_continuous=False,
    mode="classification",
    random_state=SEED,
)

# rice test sample (class 0)
rice_idx = int(np.where(y_te == 0)[0][0])
exp = lime_explainer.explain_instance(
    X_te_s[rice_idx], rf500.predict_proba,
    num_features=7, top_labels=1,
    labels=[0],
)
lime_vals = exp.as_list(label=0)   # [(feature_str, weight), ...]
lime_features = [x[0].split(" ")[0].replace(">","").replace("<","").strip()
                 for x in lime_vals]
# map back to original feature names
feature_order = []
weights = []
for feat_str, w in lime_vals:
    for f in FEATURES:
        if f in feat_str:
            feature_order.append(f)
            weights.append(w)
            break
    else:
        feature_order.append(feat_str)
        weights.append(w)

colors_lime = [P["green"] if w > 0 else P["red"] for w in weights]

fig, ax = plt.subplots(figsize=(10, 5), facecolor=P["bg"])
ax.set_facecolor(P["panel"])
for sp in ax.spines.values(): sp.set_edgecolor(P["border"])
ax.barh(range(len(weights)), weights, color=colors_lime, alpha=0.85)
ax.set_yticks(range(len(weights)))
ax.set_yticklabels(feature_order, fontsize=10, color=P["text"])
ax.set_xlabel("LIME Weight (contribution to rice class)", color=P["text"], fontsize=10)
ax.set_title("LIME Feature Contributions — Rice Prediction (RF500, augmented dataset)",
             color=P["text"], fontsize=11, fontweight="bold")
ax.tick_params(colors=P["text"])
ax.grid(axis="x", color=P["border"], linewidth=0.4)
ax.axvline(0, color=P["sub"], linewidth=0.8)
plt.tight_layout()
out = ASSETS / "lime_explanation.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print(f"  LIME saved → {out}")

print("=== SHAP ===")
# TreeExplainer on RF500
shap_explainer = shap.TreeExplainer(rf500)
# compute on a sample of test data for speed
N_SHAP = min(200, len(X_te_s))
shap_values = shap_explainer.shap_values(X_te_s[:N_SHAP])   # list of 22 arrays
# shap_values[0] → rice class, shape (N_SHAP, 7)

with open(ASSETS / "shap_values.pkl", "wb") as f:
    pickle.dump({"shap_values": shap_values, "X_te_sample": X_te_s[:N_SHAP]}, f)

# ── SHAP summary plot (beeswarm for rice) ─────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": P["bg"],
    "axes.facecolor":   P["panel"],
    "text.color":       P["text"],
    "axes.labelcolor":  P["text"],
    "xtick.color":      P["text"],
    "ytick.color":      P["text"],
})
plt.figure(facecolor=P["bg"])
shap.summary_plot(shap_values[0], X_te_s[:N_SHAP],
                  feature_names=FEATURES, show=False, plot_type="dot",
                  color_bar=True, max_display=7)
fig = plt.gcf()
fig.set_facecolor(P["bg"])
for ax in fig.get_axes():
    ax.set_facecolor(P["panel"])
out = ASSETS / "shap_summary.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print(f"  SHAP summary saved → {out}")

# ── SHAP waterfall (single rice instance) ─────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": P["bg"],
    "axes.facecolor":   P["panel"],
    "text.color":       P["text"],
})
sv_obj = shap_explainer(X_te_s[rice_idx:rice_idx+1])
plt.figure(facecolor=P["bg"])
shap.plots.waterfall(sv_obj[0, :, 0], show=False)
fig = plt.gcf()
fig.set_facecolor(P["bg"])
for ax in fig.get_axes():
    ax.set_facecolor(P["panel"])
out = ASSETS / "shap_waterfall_rice.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print(f"  SHAP waterfall saved → {out}")

# reset rcParams
plt.rcParams.update(plt.rcParamsDefault)

## 8. Confusion Matrix

In [ ]:
print("\n=== Confusion Matrix (Deep Voting Ensemble) ===")

y_pred_all = deep_voting_predict(X_te_s)
cm_arr = confusion_matrix(y_te, y_pred_all)

fig, ax = plt.subplots(figsize=(14, 12), facecolor=P["bg"])
ax.set_facecolor(P["panel"])
for sp in ax.spines.values(): sp.set_edgecolor(P["border"])

im = ax.imshow(cm_arr, cmap="Blues", aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02).ax.tick_params(colors=P["text"])

ax.set_xticks(range(22))
ax.set_yticks(range(22))
ax.set_xticklabels(CROP_NAMES, rotation=45, ha="right", fontsize=7, color=P["text"])
ax.set_yticklabels(CROP_NAMES, fontsize=7, color=P["text"])
ax.tick_params(length=0)

vmax = cm_arr.max()
for i in range(22):
    for j in range(22):
        val = cm_arr[i, j]
        if val > 0:
            tc = "white" if val > vmax * 0.55 else P["text"]
            ax.text(j, i, str(val), ha="center", va="center",
                    fontsize=5.5, color=tc, fontweight="bold")

dv_acc = accuracy_score(y_te, y_pred_all)
ax.set_title(
    f"Confusion Matrix — Deep Voting Ensemble "
    f"(Acc={dv_acc*100:.2f}%, N_test={len(y_te)})",
    color=P["text"], fontsize=11, fontweight="bold", pad=12)
ax.set_xlabel("Predicted", color=P["text"], fontsize=10)
ax.set_ylabel("True", color=P["text"], fontsize=10)

plt.tight_layout()
out = ASSETS / "confusion_matrix.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print(f"  Saved {out}")
np.save(ASSETS / "cm.npy", cm_arr)

## 9. Save All Artifacts

In [ ]:
print("\n=== Saving Artifacts ===")

with open(ASSETS / "RESULTS_12.pkl", "wb") as f:
    pickle.dump(RESULTS_12, f)
print("  RESULTS_12 saved → paper/assets/RESULTS_12.pkl")

# print final summary table
print("\n" + "=" * 70)
print(f"{'Method':<32} {'Before':>8} {'After':>8}  {'Beat baseline?'}")
print("=" * 70)
baseline = RESULTS_12["VotingClassifier\n(RF+ET)"][4]
for k, v in RESULTS_12.items():
    name = k.replace("\n", " ")
    star = " ★" if v[4] > baseline else ""
    print(f"  {name:<30} {v[0]:.4f}   {v[4]:.4f}{star}")
print("=" * 70)
print(f"  Baseline VotingClassifier(RF+ET): {baseline:.4f}")
best_name = max(RESULTS_12, key=lambda k: RESULTS_12[k][4])
best_acc  = RESULTS_12[best_name][4]
print(f"  Best model: {best_name.replace(chr(10),' ')} @ {best_acc:.4f} ({best_acc*100:.2f}%)")
print("=" * 70)

print("\n✅ All done. Figures in paper/assets/, artifacts in model/")